In [ ]:
import pandas as pd
import numpy as np
import string
import random
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix

# STAGE 1: EXCEL INPUT, CLEANING, & AUGMENTATION

# Load Excel Dataset
df = pd.read_excel('combined_dataset.xlsx', sheet_name='Sheet1')

# Drop rows with empty content or labels (NaN)
df_clean = df.dropna(subset=['konten', 'spam']).copy()
df_clean['spam'] = df_clean['spam'].str.strip().str.upper()

X_raw = df_clean['konten'].astype(str).tolist()
y = df_clean['spam'].map({'SPAM': 1, 'HAM': 0}).values

# Train-Test Split (80:20 Ratio)
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# Separate lists by class for manipulation
X_train_spam = [text for text, label in zip(X_train_raw, y_train) if label == 1]
X_train_ham = [text for text, label in zip(X_train_raw, y_train) if label == 0]

X_train_augmented = []
y_train_augmented = []

# SPAM AUGMENTATION FUNCTION (Adversarial/Obfuscation Simulation)
def apply_obfuscation(text, technique):
    text = text.lower()
    if technique == 'substitution':
        return text.translate(str.maketrans('aeio', '4310'))
    elif technique == 'insertion':
        word_list = text.split()
        new_words = ['.'.join(list(w)) if len(w) > 2 else w for w in word_list]
        return ' '.join(new_words)
    elif technique == 'vowel_removal':
        return text.translate(str.maketrans('', '', 'aeiou'))
    return text

# Exhaustive SPAM Injection (Strengthened against obfuscated text)
for text in X_train_spam:
    # Duplicate original SPAM text (insert twice) for strong baseline
    X_train_augmented.append(text); y_train_augmented.append(1)
    X_train_augmented.append(text); y_train_augmented.append(1)

    # Insert the 3 Obfuscation variations
    X_train_augmented.append(apply_obfuscation(text, 'substitution')); y_train_augmented.append(1)
    X_train_augmented.append(apply_obfuscation(text, 'insertion')); y_train_augmented.append(1)
    X_train_augmented.append(apply_obfuscation(text, 'vowel_removal')); y_train_augmented.append(1)

# DYNAMIC HAM AUGMENTATION (To prevent numerical memorization)
# Target multiplied by 5 because 1 original SPAM now produces 5 rows
target_ham_count = len(X_train_spam) * 5

for _ in range(target_ham_count):
    ham_text = random.choice(X_train_ham)

    # Inject Random Numbers and Chat Affixes (60% probability)
    if random.random() > 0.4:
        rand_num = random.randint(1, 99)
        num_injection = random.choice([f" jam {rand_num}", f" tgl {rand_num}", f" {rand_num} orang", ""])
        word_injection = random.choice([" ya", " nih", " bro", " sis", " pak", ""])
        ham_text += num_injection + word_injection

    X_train_augmented.append(ham_text)
    y_train_augmented.append(0)

# Shuffling (Randomize Data Order)
indices = np.arange(len(X_train_augmented))
np.random.shuffle(indices)
X_train_augmented = [X_train_augmented[i] for i in indices]
y_train_augmented = np.array([y_train_augmented[i] for i in indices])

# Text Cleaning (Remove emojis & symbols outside valid alphabet)
valid_alphabet = string.ascii_lowercase + string.digits + " .,?!"
def clean_text(text):
    return ''.join([char for char in text.lower() if char in valid_alphabet])

X_train_clean = [clean_text(sms) for sms in X_train_augmented]
X_test_clean = [clean_text(sms) for sms in X_test_raw]

print(f"Final Training Data Distribution: SPAM={sum(y_train_augmented)}, HAM={len(y_train_augmented)-sum(y_train_augmented)}")


# STAGE 2: SEQUENCING AND CHAR-LEVEL QUANTIZATION
MAX_LEN = 500
VOCAB_SIZE = len(valid_alphabet) + 1

# Fitting Character-Specific Tokenizer
tokenizer = Tokenizer(num_words=None, char_level=True, oov_token=None)
tokenizer.fit_on_texts([valid_alphabet])

# Creating Fixed-Length Input Matrix
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train_clean), maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(tokenizer.texts_to_sequences(X_test_clean), maxlen=MAX_LEN, padding='post', truncating='post')


# STAGE 3: CHAR-CNN MODEL ARCHITECTURE
EMBEDDING_DIM = 50

inputs = Input(shape=(MAX_LEN,), name="Input_SMS")
x = Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM, name="Embedding")(inputs)

# Parallel Conv1D (Character n-gram extraction: Trigram, 4-gram, 5-gram)
conv_3 = Conv1D(filters=128, kernel_size=3, activation='relu', name="Conv1D_3")(x)
conv_4 = Conv1D(filters=128, kernel_size=4, activation='relu', name="Conv1D_4")(x)
conv_5 = Conv1D(filters=128, kernel_size=5, activation='relu', name="Conv1D_5")(x)

# Max Pooling
pool_3 = GlobalMaxPooling1D(name="GMP_3")(conv_3)
pool_4 = GlobalMaxPooling1D(name="GMP_4")(conv_4)
pool_5 = GlobalMaxPooling1D(name="GMP_5")(conv_5)

# Dense, Dropout & Final Classification
merged_features = concatenate([pool_3, pool_4, pool_5])
dense = Dense(256, activation='relu')(merged_features)
dropout = Dropout(0.5)(dense)
outputs = Dense(1, activation='sigmoid')(dropout)

model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Early Stopping (Lock model at best iteration)
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

print("\n--- Starting Model Training Process ---")
history = model.fit(
    X_train_pad, y_train_augmented,
    epochs=15,
    batch_size=32,
    validation_data=(X_test_pad, y_test),
    callbacks=[early_stopping],
    verbose=1
)


# STAGE 4: FINAL TESTING & CROSS-TESTING SCENARIOS
print("\n--- Model Evaluation on Test Set ---")
y_pred = (model.predict(X_test_pad) > 0.5).astype(int)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['NON-SPAM (HAM)', 'SPAM']))

# Scenario 2: Obfuscation Robustness Test (Hard Scenarios)
test_messages = [
    "Selamat, nomor anda memenangkan hadiah uang tunai 10 juta. Hubungi bit.ly/hadiah",
    "S3lamat, n0m0r 4nd4 m3m3n4ngk4n h4d14h u4ng tun41 10 jut4. Hubung1 b1t.ly/h4d14h",
    "S.e.l.a.m.a.t n.o.m.o.r anda m.e.m.e.n.a.n.g.k.a.n hadiah. Hubungi kami.",
    "Besok jangan lupa kumpul tugas alpro jam 10 pagi di ruang dosen ya."
]

X_test_scenario_pad = pad_sequences(tokenizer.texts_to_sequences([clean_text(t) for t in test_messages]), maxlen=MAX_LEN, padding='post', truncating='post')
scenario_probs = model.predict(X_test_scenario_pad)

print("\n==================================================")
print("CHAR-CNN MODEL TEST RESULTS (OBFUSCATION SCENARIOS)")
print("==================================================")
for i, text in enumerate(test_messages):
    prob = scenario_probs[i][0]
    pred = "SPAM" if prob > 0.5 else "NON-SPAM (HAM)"
    print(f"Message      : {text}")
    print(f"Probability  : {prob:.4f}")
    print(f"Prediction   : {pred}")
    print("-" * 50)